# Pandas 第4课：分组聚合与多表合并

> 数据分析最常见的问题："每个部门/每类产品 的总额、均值是多少？"
> 这就是 **groupby（分组聚合）**，对应 SQL 的 GROUP BY。
> 本课还讲多表合并 merge/concat 和透视表。
>
> 参考：Pandas 官方 groupby（split-apply-combine）、清华 Pandas 讲义。

In [1]:
import pandas as pd
import numpy as np

orders=pd.DataFrame({
    '部门':['销售','销售','技术','技术','行政','销售','技术'],
    '产品':['A','B','A','B','A','B','A'],
    '销售员':['张三','李四','王五','赵六','钱七','张三','王五'],
    '销售额':[800,1200,650,1500,400,950,700],
    '数量':[8,12,6,15,4,9,7]
})
print(orders)

   部门 产品 销售员   销售额  数量
0  销售  A  张三   800   8
1  销售  B  李四  1200  12
2  技术  A  王五   650   6
3  技术  B  赵六  1500  15
4  行政  A  钱七   400   4
5  销售  B  张三   950   9
6  技术  A  王五   700   7


## 1. groupby：split-apply-combine 三步

- **Split 拆分**：按某列把表分成若干组
- **Apply 应用**：每组分别聚合（sum/mean/count...）
- **Combine 合并**：结果拼回一张表

```python
df.groupby('部门')['销售额'].sum()
```

In [2]:
print(orders.groupby('部门')['销售额'].sum())
print(orders.groupby('部门', as_index=False)['销售额'].mean())

部门
技术    2850
行政     400
销售    2950
Name: 销售额, dtype: int64
   部门         销售额
0  技术  950.000000
1  行政  400.000000
2  销售  983.333333


#### 易错点：sum 也会"拼"字符串，不是数值专属

- pandas **2.x** 里 `df.groupby('列').sum()` 默认连文字列也拼接（字符串 `+` 即拼接），如产品 `A+B+A -> ABA`；老版本默认只算数值列，注意版本差异
- 只想算数值列：`.sum(numeric_only=True)`
- 裸拼没有分隔符、糊成一团不可读；字符串聚合的正确姿势：
  - `.apply(lambda x: ','.join(x))`：加分隔符拼接
  - `.apply(list)`：收集成列表
  - 常用：`.first()/.last()/.nunique()/.count()/.mode()`


### 多列分组 / 一次多个聚合

- 按多列分组得到多层索引
- `.agg([...])` 一次算多个统计

In [3]:
print(orders.groupby(['部门','产品'])['销售额'].sum())
print(orders.groupby('部门')['销售额'].agg(['sum','mean','count']).round(2))

部门  产品
技术  A     1350
    B     1500
行政  A      400
销售  A      800
    B     2150
Name: 销售额, dtype: int64
     sum    mean  count
部门                     
技术  2850  950.00      3
行政   400  400.00      1
销售  2950  983.33      3


### 命名聚合（结果列名自定义，推荐）

In [4]:
r=orders.groupby('部门').agg(
    总销售额=('销售额','sum'),
    平均销售额=('销售额','mean'),
    总数量=('数量','sum'))
print(r.round(2))

    总销售额   平均销售额  总数量
部门                   
技术  2850  950.00   28
行政   400  400.00    4
销售  2950  983.33   29


## 2. transform：组内运算，返回和原表一样长

聚合会"压缩"行数；**transform 不压缩**，把组级结果"广播"回每一行。
典型用途：组内占比、组内标准化、按组均值填充缺失。

In [5]:
orders['部门总额']=orders.groupby('部门')['销售额'].transform('sum')
orders['部门内占比']=(orders['销售额']/orders['部门总额']).round(2)
print(orders)

   部门 产品 销售员   销售额  数量  部门总额  部门内占比
0  销售  A  张三   800   8  2950   0.27
1  销售  B  李四  1200  12  2950   0.41
2  技术  A  王五   650   6  2850   0.23
3  技术  B  赵六  1500  15  2850   0.53
4  行政  A  钱七   400   4   400   1.00
5  销售  B  张三   950   9  2950   0.32
6  技术  A  王五   700   7  2850   0.25


## 3. filter：按组条件筛选整组

保留"满足条件的整个组"（不是删组内某些行）。

In [6]:
print(orders.groupby('部门').filter(lambda g: g['销售额'].sum()>2900))
print(orders[orders['部门总额']>2900])

   部门 产品 销售员   销售额  数量  部门总额  部门内占比
0  销售  A  张三   800   8  2950   0.27
1  销售  B  李四  1200  12  2950   0.41
5  销售  B  张三   950   9  2950   0.32
   部门 产品 销售员   销售额  数量  部门总额  部门内占比
0  销售  A  张三   800   8  2950   0.27
1  销售  B  李四  1200  12  2950   0.41
5  销售  B  张三   950   9  2950   0.32


## 4. 排序与计数

- `sort_values(列, ascending=)`：排序（多列传列表）
- `value_counts()`：每个值出现次数（分组计数快捷方式）

In [7]:
print(orders.sort_values('销售额', ascending=False))
print(orders['部门'].value_counts())

   部门 产品 销售员   销售额  数量  部门总额  部门内占比
3  技术  B  赵六  1500  15  2850   0.53
1  销售  B  李四  1200  12  2950   0.41
5  销售  B  张三   950   9  2950   0.32
0  销售  A  张三   800   8  2950   0.27
6  技术  A  王五   700   7  2850   0.25
2  技术  A  王五   650   6  2850   0.23
4  行政  A  钱七   400   4   400   1.00
部门
销售    3
技术    3
行政    1
Name: count, dtype: int64


## 5. 多表合并：merge（重点）

再造一张部门信息表，练习"关系连接"。

In [8]:
dept=pd.DataFrame({
    '部门':['销售','技术','行政','财务'],
    '地点':['上海','北京','广州','深圳']})
print(dept)

   部门  地点
0  销售  上海
1  技术  北京
2  行政  广州
3  财务  深圳


```python
pd.merge(左表, 右表, on='共同列', how='连接方式')
```

| how | 含义（类似SQL） |
|---|---|
| `inner` | 只保留两边都有的（默认） |
| `left` | 左表全保留，右表匹配不到为NaN |
| `right` | 右表全保留 |
| `outer` | 两边都保留 |

> 列名不同时用 `left_on/right_on`。

In [9]:
print(pd.merge(orders, dept, on='部门', how='left'))
print(pd.merge(orders, dept, on='部门', how='outer'))

   部门 产品 销售员   销售额  数量  部门总额  部门内占比  地点
0  销售  A  张三   800   8  2950   0.27  上海
1  销售  B  李四  1200  12  2950   0.41  上海
2  技术  A  王五   650   6  2850   0.23  北京
3  技术  B  赵六  1500  15  2850   0.53  北京
4  行政  A  钱七   400   4   400   1.00  广州
5  销售  B  张三   950   9  2950   0.32  上海
6  技术  A  王五   700   7  2850   0.25  北京
   部门   产品  销售员     销售额    数量    部门总额  部门内占比  地点
0  技术    A   王五   650.0   6.0  2850.0   0.23  北京
1  技术    B   赵六  1500.0  15.0  2850.0   0.53  北京
2  技术    A   王五   700.0   7.0  2850.0   0.25  北京
3  行政    A   钱七   400.0   4.0   400.0   1.00  广州
4  财务  NaN  NaN     NaN   NaN     NaN    NaN  深圳
5  销售    A   张三   800.0   8.0  2950.0   0.27  上海
6  销售    B   李四  1200.0  12.0  2950.0   0.41  上海
7  销售    B   张三   950.0   9.0  2950.0   0.32  上海


## 6. concat：直接拼接（不看值，按位置堆）

- `axis=0`：上下堆（加行，最常用）
- `axis=1`：左右拼（加列）
> merge 是"按共同列匹配"，concat 是"硬拼"。

In [10]:
a=orders.iloc[:2]; b=orders.iloc[2:4]
c=orders[:2]
print(a)
print(c)
print(pd.concat([a,b], ignore_index=True))

   部门 产品 销售员   销售额  数量  部门总额  部门内占比
0  销售  A  张三   800   8  2950   0.27
1  销售  B  李四  1200  12  2950   0.41
   部门 产品 销售员   销售额  数量  部门总额  部门内占比
0  销售  A  张三   800   8  2950   0.27
1  销售  B  李四  1200  12  2950   0.41
   部门 产品 销售员   销售额  数量  部门总额  部门内占比
0  销售  A  张三   800   8  2950   0.27
1  销售  B  李四  1200  12  2950   0.41
2  技术  A  王五   650   6  2850   0.23
3  技术  B  赵六  1500  15  2850   0.53


## 7. 透视表与长宽转换

### pivot_table：Excel 透视表
```python
pd.pivot_table(df, values='销售额', index='部门',
               columns='产品', aggfunc='sum', fill_value=0,
               margins=True)   # margins 加合计
```

In [11]:
pv=pd.pivot_table(orders, values='销售额', index='部门',
                 columns='产品', aggfunc='sum', fill_value=0, margins=True)
print(pv)

产品      A     B   All
部门                   
技术   1350  1500  2850
行政    400     0   400
销售    800  2150  2950
All  2550  3650  6200


### melt：宽表变长表（透视的逆操作）

In [12]:
wide=pd.DataFrame({'部门':['销售','技术'],'一月':[100,200],'二月':[300,400]})
print(wide)
print(wide.melt(id_vars='部门', var_name='月份', value_name='销售额'))

   部门   一月   二月
0  销售  100  300
1  技术  200  400
   部门  月份  销售额
0  销售  一月  100
1  技术  一月  200
2  销售  二月  300
3  技术  二月  400


## 8. 练习

用 `orders` 和 `dept` 完成：
1. 统计每个部门的总销售额、订单笔数
2. 统计每个销售员的总销售额并降序排列
3. 做一张透视表：行=部门、列=产品、值=数量合计
4. 新增列：每笔订单在本部门销售额中的占比
5. 用 merge 给 orders 加上部门地点，找出地点在上海的订单
6. 找出总销售额最高的部门（提示：groupby+sum+idxmax）

In [13]:
# 在此完成
# 题1：订单"笔数"要用 count（数有几笔），不是数量的 sum
print(orders.groupby('部门').agg(
    总销售额=('销售额','sum'),
    订单笔数=('销售额','count')))

# 题2：每个销售员总销售额，降序
print(orders.groupby('销售员')['销售额'].sum().sort_values(ascending=False))

# 题3：透视表，题目要"数量合计"，values='数量'
pv = pd.pivot_table(orders, values='数量', index='部门', columns='产品',
                    aggfunc='sum', fill_value=0, margins=True)
print(pv)

# 题4：本部门内销售额占比，用 transform（filter 只做判断、不参与计算）
orders['销售额占比'] = (orders['销售额'] /
        orders.groupby('部门')['销售额'].transform('sum')).round(2)
print(orders)

# 题5：merge 加地点，筛上海
orders = pd.merge(orders, dept, on='部门', how='left')
print(orders[orders['地点']=='上海'])
print(orders[orders['地点'].isin(['上海','北京'])])

# 题6：总销售额最高部门，idxmax 直接返回标签
s = orders.groupby('部门')['销售额'].sum()
print("最高部门:", s.idxmax(), s.max())


    总销售额  订单笔数
部门            
技术  2850     3
行政   400     1
销售  2950     3
销售员
张三    1750
赵六    1500
王五    1350
李四    1200
钱七     400
Name: 销售额, dtype: int64
产品    A   B  All
部门              
技术   13  15   28
行政    4   0    4
销售    8  21   29
All  25  36   61
   部门 产品 销售员   销售额  数量  部门总额  部门内占比  销售额占比
0  销售  A  张三   800   8  2950   0.27   0.27
1  销售  B  李四  1200  12  2950   0.41   0.41
2  技术  A  王五   650   6  2850   0.23   0.23
3  技术  B  赵六  1500  15  2850   0.53   0.53
4  行政  A  钱七   400   4   400   1.00   1.00
5  销售  B  张三   950   9  2950   0.32   0.32
6  技术  A  王五   700   7  2850   0.25   0.25
   部门 产品 销售员   销售额  数量  部门总额  部门内占比  销售额占比  地点
0  销售  A  张三   800   8  2950   0.27   0.27  上海
1  销售  B  李四  1200  12  2950   0.41   0.41  上海
5  销售  B  张三   950   9  2950   0.32   0.32  上海
   部门 产品 销售员   销售额  数量  部门总额  部门内占比  销售额占比  地点
0  销售  A  张三   800   8  2950   0.27   0.27  上海
1  销售  B  李四  1200  12  2950   0.41   0.41  上海
2  技术  A  王五   650   6  2850   0.23   0.23  北京
3  技术  B  赵六  1500  

#### 我的原始答案（留作对照，错误处已标注）

```python
# 在此完成
print(orders)
print(orders.groupby('部门').agg(
    总销售额=('销售额','sum'),
    订单数量=('数量','sum')))   # ❌ 题1：订单笔数要用 count，这里算成了"商品数量合计"
print(orders.groupby('销售员').agg(总销售额=('销售额','sum')).sort_values('总销售额', ascending=False))  # ✓题2
pv = pd.pivot_table(orders, values='销售额', index='部门', columns='产品',
                    aggfunc='sum', fill_value=0, margins=True)  # ❌ 题3：题目要数量，values 应填 '数量'
print(pv)
orders['销售额占比'] = (orders['销售额'] /
        orders.groupby('部门')['销售额'].transform('sum')).round(2)  # ✓题4
print(orders)
orders = pd.merge(orders, dept, on='部门', how='left')  # ✓题5
print(orders[orders['地点'] == '上海'])
print(orders[orders['地点'].isin(['上海', '北京'])])
print(orders.groupby('部门')['销售额'].sum().sort_values(ascending=False).head(1))  # 题6结果✓，可简化为 idxmax
```

**两处错误的根子：都没盯紧题目要的"哪个指标 / 哪一列"。**


### 批改

**对：题2、题4、题5。** 题4 你那句注释特别到位，"filter 用来判断、transform 才参与计算"，说明这两个方法你真分清了；题5 主动用 `isin` 扩展多值筛选，也很好。

**错两处（都是没按题目指定的列/指标）：**
- **题1：订单笔数要用 `count`，你写成了 `('数量','sum')`。**
  - count 数"有几笔订单"（技术3、行政1、销售3）
  - 数量 sum 是"商品卖了多少件"，是另一个指标，别和笔数混
- **题3：题目明确要"值=数量合计"，你写了 `values='销售额'`**，应改成 `values='数量'`

**题6：你的 `sort_values+head(1)` 结果正确**，是可行写法；更规范的是 `s.idxmax()` 直接拿到最高部门标签、不用整体排序（代码更短、数据大时更快）。

#### 易错点提醒
- "笔数/人数/记录数" → `count`（或 agg 里 `('列','count')`）
- 做 pivot_table 先确认题目要的是哪一列，`values=` 别填错
- 找"最大/最小对应的那个标签"用 `.idxmax()/.idxmin()`，要取值配 `.max()/.min()`


### 小结

- groupby：拆分-应用-合并；`agg` 多统计、命名聚合
- transform：组级结果广播回每行（占比/标准化/填充）
- filter：按组条件保留整组
- merge：按共同列匹配，how 控制 inner/left/right/outer
- concat：硬拼接（axis 0加行/1加列）
- pivot_table 透视、melt 逆透视
- 下一份：Pandas 综合练习卷（纯题目）